In [1]:
# !pip install -r requirement.txt

In [2]:
import numpy as np
import pandas as pd
import torch
from torch import nn
import torchaudio
from safetensors.torch import save_file
from model_audio_input.model import SpecTTTra
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, WeightedRandomSampler
from model_audio_input.dataset import SonicDataset

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
hyper_params = [
    { # alpha
        "input_spec_dim": 128,
        "input_temp_dim": 188,
        "embed_dim": 384,
        "f_clip": 1,
        "t_clip": 3,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # beta
        "input_spec_dim": 128,
        "input_temp_dim": 188,
        "embed_dim": 384,
        "f_clip": 3,
        "t_clip": 5,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    },
    { # gamma
        "input_spec_dim": 128,
        "input_temp_dim": 188,
        "embed_dim": 384,
        "f_clip": 5,
        "t_clip": 7,
        "num_heads": 6,
        "num_layers": 12,
        "pos_drop_rate": 0
    }
]

In [14]:
df = pd.read_csv("crop_data/crop6.csv")
data = SonicDataset(df, duration_seconds=6.0)

train_loader = DataLoader(data, batch_size=32, shuffle=True, num_workers=2)
batch = next(iter(train_loader))
X = batch["x"]
y = batch["y"]
print(X.shape, y.shape)

num_zeros = (df["label"] == 0).sum().item()
num_ones = (df["label"] == 1).sum().item()

print("Số label 0:", num_zeros)
print("Số label 1:", num_ones)


/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)


torch.Size([32, 96000]) torch.Size([32])
Số label 0: 7000
Số label 1: 5235


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12235 entries, 0 to 12234
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   filename   12235 non-null  object
 1   path       12235 non-null  object
 2   fake_type  12235 non-null  object
 3   label      12235 non-null  int64 
dtypes: int64(1), object(3)
memory usage: 382.5+ KB


In [7]:
train_loss_hist, train_acc_hist, test_acc_hist = [], [], []
num_epoch = 10

In [15]:
def train(len_clip=6, version=1, lr=1e-5, weight_decay=1e-4, use_balanced_sampler=True):
    if len_clip != 6:
        raise ValueError("This training pipeline is configured for 6s clips only.")
    if version not in [1, 2, 3]:
        raise ValueError("Not available model version")

    hyper_param = hyper_params[version - 1].copy()
    hyper_param["input_temp_dim"] = 188

    df = pd.read_csv("crop_data/crop6.csv")

    df_train, df_test = train_test_split(
        df,
        test_size=0.2,
        stratify=df["label"],
        random_state=42
    )

    print(f"Train data: {df_train.shape}")
    print(f"Test_data: {df_test.shape}")

    train_loss_hist.clear()
    train_acc_hist.clear()
    test_acc_hist.clear()

    train_data = SonicDataset(df_train, duration_seconds=6.0)
    test_data = SonicDataset(df_test, duration_seconds=6.0)

    train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

    model = SpecTTTra(
        **hyper_param,
        expected_samples=train_data.expected_samples,
    ).to(device)

    loss = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )
    
    for epoch in range(num_epoch):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total_sample = 0
        train_pred_count = torch.zeros(2, dtype=torch.long)

        for batch in train_loader:
            X = batch["x"].to(device)
            y = batch["y"].to(device).long()

            optimizer.zero_grad()

            logits = model(X)
            cur_loss = loss(logits, y)
            cur_loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += cur_loss.item() * X.size(0)
            total_sample += X.size(0)

            pred = torch.argmax(logits, dim=1)
            total_correct += (pred == y).sum().item()
            train_pred_count += torch.bincount(pred.detach().cpu(), minlength=2)

        epoch_loss = total_loss / total_sample
        epoch_acc = total_correct / total_sample

        model.eval()
        test_sample = 0
        test_correct = 0
        test_pred_count = torch.zeros(2, dtype=torch.long)
        test_cm = torch.zeros((2, 2), dtype=torch.long)
        with torch.no_grad():
            for batch in test_loader:
                X = batch["x"].to(device)
                y = batch["y"].to(device).long()

                logits = model(X)
                pred = torch.argmax(logits, dim=1)
                test_sample += X.size(0)
                test_correct += (y == pred).sum().item()
                test_pred_count += torch.bincount(pred.detach().cpu(), minlength=2)
                test_cm += torch.bincount((y.detach().cpu() * 2 + pred.detach().cpu()), minlength=4).reshape(2, 2)

        test_acc = test_correct / test_sample
        recall_fake = test_cm[0, 0].item() / max(1, test_cm[0].sum().item())
        recall_real = test_cm[1, 1].item() / max(1, test_cm[1].sum().item())
        balanced_acc = (recall_fake + recall_real) / 2.0

        train_loss_hist.append(epoch_loss)
        train_acc_hist.append(epoch_acc)
        test_acc_hist.append(test_acc)

        print(
            f"Epoch [{epoch+1}/{num_epoch}] "
            f"Train Loss: {epoch_loss:.4f} | "
            f"Train Acc: {epoch_acc:.4f} | "
            f"Test Acc: {test_acc:.4f} | "
            f"Bal Acc: {balanced_acc:.4f} | "
            f"Recall[Fake/Real]: [{recall_fake:.4f}, {recall_real:.4f}] | "
            f"Train Pred[0/1]: {train_pred_count.tolist()} | "
            f"Test Pred[0/1]: {test_pred_count.tolist()} | "
            f"Test CM [[TN,FP],[FN,TP]]: {test_cm.tolist()}"
        )
    return model


In [16]:
model = train()
save_file(model.state_dict(), "model.safetensors")
print("Saved model.safetensors to workspace root")

Train data: (9788, 4)
Test_data: (2447, 4)


/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommend

Epoch [1/10] Train Loss: 0.0482 | Train Acc: 0.9811 | Test Acc: 0.9922 | Bal Acc: 0.9913 | Recall[Fake/Real]: [0.9979, 0.9847] | Train Pred[0/1]: [5615, 4173] | Test Pred[0/1]: [1413, 1034] | Test CM [[TN,FP],[FN,TP]]: [[1397, 3], [16, 1031]]


/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommend

Epoch [2/10] Train Loss: 0.0171 | Train Acc: 0.9955 | Test Acc: 0.9947 | Bal Acc: 0.9948 | Recall[Fake/Real]: [0.9943, 0.9952] | Train Pred[0/1]: [5594, 4194] | Test Pred[0/1]: [1397, 1050] | Test CM [[TN,FP],[FN,TP]]: [[1392, 8], [5, 1042]]


...

Epoch [8/10] Train Loss: 0.0023 | Train Acc: 0.9990 | Test Acc: 0.9963 | Bal Acc: 0.9961 | Recall[Fake/Real]: [0.9979, 0.9943] | Train Pred[0/1]: [5596, 4192] | Test Pred[0/1]: [1403, 1044] | Test CM [[TN,FP],[FN,TP]]: [[1397, 3], [6, 1041]]


/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommend

Epoch [9/10] Train Loss: 0.0042 | Train Acc: 0.9991 | Test Acc: 0.9967 | Bal Acc: 0.9970 | Recall[Fake/Real]: [0.9950, 0.9990] | Train Pred[0/1]: [5597, 4191] | Test Pred[0/1]: [1394, 1053] | Test CM [[TN,FP],[FN,TP]]: [[1393, 7], [1, 1046]]


/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  waveform = torch.tensor(waveform, dtype=torch.float32)
/home/admin/model_audio_input/dataset.py:37: UserWarning: To copy construct from a tensor, it is recommend

Epoch [10/10] Train Loss: 0.0064 | Train Acc: 0.9988 | Test Acc: 0.9963 | Bal Acc: 0.9963 | Recall[Fake/Real]: [0.9964, 0.9962] | Train Pred[0/1]: [5602, 4186] | Test Pred[0/1]: [1399, 1048] | Test CM [[TN,FP],[FN,TP]]: [[1395, 5], [4, 1043]]
Saved model.safetensors to workspace root
